In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

In [1]:
def transition_probs_from_map(map, start_state):
    n_rows = len(map)
    n_cols = len(map[0])
    n_states = n_rows * n_cols
    n_actions = 4 # left, up, right, down
    action_neighbors = [(0, -1), (-1, 0), (0, 1), (1, 0)]
    T = np.zeros((n_states, n_states, n_actions))
    R = np.zeros((n_states))
    for row_idx, row in enumerate(map):
        for cell_idx, cell in enumerate(row):
            state_idx = row_idx * n_cols + cell_idx
            if cell > 1:
                R[state_idx] = cell
            for action in range(n_actions):
                if cell > 1:
                    T[state_idx, start_state, action] = 1.
                else:
                    target_row = row_idx + action_neighbors[action][0]
                    target_column = cell_idx + action_neighbors[action][1]
                    if target_row < 0 or target_row >= n_rows or target_column < 0 or target_column >= n_cols:
                        T[state_idx, state_idx, action] = 1.
                    else:
                        target_resistance = map[target_row][target_column] if map[target_row][target_column] <= 1 else 0
                        T[state_idx, state_idx, action] = target_resistance
                        target_state = target_row * n_cols + target_column
                        T[state_idx, target_state, action] = 1. - target_resistance
    return T, R

def Q_learning(T, R, gamma, max_iter, alpha, epsilon, init_state, Q_init=None):
    n_states = R.shape[0]
    n_actions = T.shape[2]
    states = list(range(n_states))
    actions = list(range(n_actions))
    Q = np.zeros((n_states, n_actions))
    if not Q_init is None:
        Q = np.copy(Q_init)
    value_sums = [0]
    act_state = init_state
    for i in range(max_iter):
        if random.uniform(0, 1) < epsilon:
            action = random.choice(actions)
        else:
            best_actions = np.argwhere(Q[act_state, :] == np.amax(Q[act_state, :])).flatten().tolist()
            action = random.choice(best_actions)

        #print(T[act_state, :, action].flatten())
        next_state = np.random.choice(states, size=1, p=T[act_state, :, action].flatten())
        reward = R[next_state]
        TD = reward + gamma * Q[next_state, :].max()
        Q[act_state, action] = (1 - alpha) * Q[act_state, action] + alpha * TD

        act_state = next_state
        value_sums.append(Q.sum() / n_states)
    return value_sums, Q